## Import Polars

In [49]:
# pip install polars
# poetry add polars
# uv add polars

In [50]:
import polars as pl

## Read Data into DataFrame

In [51]:
COLUMN_NAMES = [
        "id", "price", "date", "postcode", "property_type",
        "old_new", "duration", "paon", "saon", "street",
        "locality", "town_city", "district", "county",
        "ppd_category", "record_type",
    ]

In [52]:
land_registry_data = pl.read_csv(
    source="../../data/land_registry_data/pp-*.csv",
    has_header=False,
    new_columns=COLUMN_NAMES,
    infer_schema=True,
    null_values=[""],
)

In [53]:
land_registry_data.sample(5)

id,price,date,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_type
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""{CB0035E6-7BEB-58AE-E053-6B04A…",33500,"""2021-05-14 00:00""","""DL17 8HZ""","""T""","""N""","""F""","""3""",null,"""HALLGARTH TERRACE""",null,"""FERRYHILL""","""COUNTY DURHAM""","""COUNTY DURHAM""","""B""","""A"""
"""{C6209F5F-B1D7-295E-E053-6C04A…",579950,"""2021-06-04 00:00""","""OX25 4RR""","""T""","""N""","""F""","""NEW MANOR HOUSE""","""2""","""SOUTH SIDE""","""STEEPLE ASTON""","""BICESTER""","""CHERWELL""","""OXFORDSHIRE""","""A""","""A"""
"""{E2D14905-A867-4C2D-E053-6B04A…",240000,"""2022-04-29 00:00""","""LN2 4GJ""","""D""","""N""","""F""","""1""",null,"""FLAXLEY CLOSE""",null,"""LINCOLN""","""LINCOLN""","""LINCOLNSHIRE""","""A""","""A"""
"""{D707E535-C28C-0AD9-E053-6B04A…",252000,"""2021-08-06 00:00""","""DL8 1NH""","""T""","""N""","""F""","""2""",null,"""CHURCH VIEW""","""HORNBY""","""BEDALE""","""RICHMONDSHIRE""","""NORTH YORKSHIRE""","""A""","""A"""
"""{85866A65-8C49-143F-E053-6B04A…",585000,"""2017-11-30 00:00""","""BH13 7EZ""","""O""","""N""","""F""","""ELMO, 9""",null,"""ELMSTEAD ROAD""","""CANFORD CLIFFS""","""POOLE""","""POOLE""","""POOLE""","""B""","""A"""


In [54]:
land_registry_data.describe()

statistic,id,price,date,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_type
str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""","""10002480""",1.000248e7,"""10002480""","""9967085""","""10002480""","""10002480""","""10002480""","""10002480""","""1351070""","""9821427""","""3800548""","""10002480""","""10002480""","""10002480""","""10002480""","""10002480"""
"""null_count""","""0""",0.0,"""0""","""35395""","""0""","""0""","""0""","""0""","""8651410""","""181053""","""6201932""","""0""","""0""","""0""","""0""","""0"""
"""mean""",null,372035.514483,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",null,1.6258e6,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""{01EB45EF-612C-40F3-E063-4704A…",1.0,"""2016-01-01 00:00""","""AL1 1AJ""","""D""","""N""","""F""","""'LEONARD' HOGARTH HOUSE, 32""","""'THE CAR PARK'-BASEMENT LEVELS…","""10TH AVENUE""","""AB KETTLEBY""","""ABBOTS LANGLEY""","""ADUR""","""BATH AND NORTH EAST SOMERSET""","""A""","""A"""
"""25%""",null,160000.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,250000.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,390000.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""{FFA361DB-9CCE-8A03-E053-4804A…",9e8,"""2025-12-31 00:00""","""YO91 1RT""","""T""","""Y""","""L""","""ZYTEK ENGINEERING LTD""","""ZURICH HOUSE 226""","""ZYBURN COURT""","""ZOUCH""","""YSTRAD MEURIG""","""YORK""","""YORK""","""B""","""A"""


In [55]:
land_registry_data = (
    land_registry_data
    .with_columns(
        pl.col("date").str.to_date(format="%Y-%m-%d %H:%M"),
    )
)

In [56]:
land_registry_data = land_registry_data.with_columns(
    pl.col("property_type").replace({
        "D": "Detached",
        "S": "Semi-Detached",
        "T": "Terraced",
        "F": "Flat",
        "O": "Other",
    }).alias("property_type")
)

In [57]:
land_registry_data["property_type"].value_counts()

property_type,count
str,u32
"""Other""",582196
"""Detached""",2303875
"""Flat""",1794725
"""Semi-Detached""",2625078
"""Terraced""",2696606


In [58]:
land_registry_data = land_registry_data.filter(pl.col("property_type") != "Other")

In [59]:
land_registry_data = (
    land_registry_data
    .with_columns(
        pl.col("date").dt.year().alias("year"),
    )
)

In [60]:
land_registry_data.select(["id", "date", "year", "property_type", "price"])

id,date,year,property_type,price
str,date,i32,str,i64
"""{3E0330F0-76C8-8D89-E050-A8C06…",2016-09-21,2016,"""Flat""",103750
"""{3E0330F0-76C9-8D89-E050-A8C06…",2016-09-16,2016,"""Semi-Detached""",165000
"""{3E0330F0-76CA-8D89-E050-A8C06…",2016-09-02,2016,"""Semi-Detached""",90000
"""{3E0330F0-76CB-8D89-E050-A8C06…",2016-08-22,2016,"""Terraced""",105000
"""{3E0330F0-76CC-8D89-E050-A8C06…",2016-08-30,2016,"""Semi-Detached""",67500
…,…,…,…,…
"""{36A61A95-56AD-DEF2-E063-4704A…",2025-05-22,2025,"""Semi-Detached""",267500
"""{36A61A95-56AE-DEF2-E063-4704A…",2025-04-29,2025,"""Semi-Detached""",230000
"""{36A61A95-56B0-DEF2-E063-4704A…",2025-04-30,2025,"""Flat""",177500


In [61]:
annual_price_by_property_type = (
    land_registry_data
    .group_by(["year", "property_type"])
    .agg(
        pl.col("price").median().alias("median_price")
    )
    .sort(["year", "property_type"])
)

In [62]:
annual_price_by_property_type

year,property_type,median_price
i32,str,f64
2016,"""Detached""",306000.0
2016,"""Flat""",200000.0
2016,"""Semi-Detached""",187500.0
2016,"""Terraced""",169000.0
2017,"""Detached""",322500.0
…,…,…
2024,"""Terraced""",220000.0
2025,"""Detached""",416100.5
2025,"""Flat""",230000.0


## Visualise the results

In [63]:
import plotly.express as px

In [64]:
px.line(
    annual_price_by_property_type,
    x="year",
    y="median_price",
    color="property_type",
    markers=True,
    title="Median Price by Year and Property Type",
    width=800,
    height=600,
)

In [65]:
annual_price_by_property_type.write_delta("../../data/price_paid_insights/annual_price_by_property_type", mode="overwrite")

In [66]:
lazy_frame = (
    pl.scan_csv(
    source="../../data/land_registry_data/pp-*.csv",
    has_header=False,
    new_columns=COLUMN_NAMES,
    infer_schema=True,
    null_values=[""])
    .with_columns(
        pl.col("date").str.to_date(format="%Y-%m-%d %H:%M"),
    )
    .with_columns(
        pl.col("property_type").replace({
            "D": "Detached",
            "S": "Semi-Detached",
            "T": "Terraced",
            "F": "Flat",
            "O": "Other",
        }).alias("property_type")
    )
    .filter(pl.col("property_type") != "Other")
    .with_columns(
        pl.col("date").dt.year().alias("year")
    )
    .group_by(["year", "property_type"])
    .agg(
        pl.col("price").median().alias("median_price")
    )
    .sort(["year", "property_type"])
)

In [67]:
lazy_frame

In [68]:
print(lazy_frame.explain())

SORT BY [col("year"), col("property_type")]
  AGGREGATE[maintain_order: false]
    [col("price").median().alias("median_price")] BY [col("year"), col("property_type")]
    FROM
     WITH_COLUMNS:
     [col("date").dt.year().alias("year")] 
      FILTER [(col("property_type")) != ("Other")]
      FROM
         WITH_COLUMNS:
         [col("date").str.strptime(["raise"]), col("property_type").replace([["D", "S", … "O"], ["Detached", "Semi-Detached", … "Other"]])] 
          Csv SCAN [../../data/land_registry_data/pp-2016.csv, ... 9 other sources]
          PROJECT 3/16 COLUMNS
          ESTIMATED ROWS: 1066856


In [69]:
lazy_frame.collect()

year,property_type,median_price
i32,str,f64
2016,"""Detached""",306000.0
2016,"""Flat""",200000.0
2016,"""Semi-Detached""",187500.0
2016,"""Terraced""",169000.0
2017,"""Detached""",322500.0
…,…,…
2024,"""Terraced""",220000.0
2025,"""Detached""",416100.5
2025,"""Flat""",230000.0


In [70]:
# Stream data to parquet
lazy_frame.sink_parquet("../../data/price_paid_insights/land_registry_data.parquet")